# NFE vs solver comparison

Compare Euler / Heun / RK4 at matched **NFE budget** rather than matched
step count, on a fixed trained model from the baseline. Lower endpoint
$W_2^2$ at low NFE is the diagnostic of interest for FM.

In [ ]:
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from otfm.couplings import hungarian_pairing
from otfm.datasets import sample_8gaussians, sample_moons
from otfm.metrics import empirical_w2_squared_hungarian
from otfm.model import init_mlp_params
from otfm.runtime import make_t_schedule, train_on_fixed_pairs
from otfm.solvers import SOLVERS, nfe_per_step


In [ ]:
cfg = yaml.safe_load(open('../configs/base.yaml'))
n = cfg['data']['n']
widths = cfg['model']['widths']
train_steps = cfg['training']['train_steps']

key = jax.random.PRNGKey(0)
key, k0, k1, kinit = jax.random.split(key, 4)
x0 = sample_moons(k0, n)
x1 = sample_8gaussians(k1, n)
base_params = init_mlp_params(kinit, widths)
t_schedule = make_t_schedule(seed=123, steps=train_steps, n=n)

xa, xb = hungarian_pairing(x0, x1)
params_tr, _ = train_on_fixed_pairs(base_params, xa, xb, t_schedule)


In [ ]:
# Compare at matched NFE: pick steps so that NFE = step * cost(solver) is constant.
target_nfes = [4, 8, 16, 32, 64]

rows = []
for solver_name, solver_fn in SOLVERS.items():
    cost = nfe_per_step(solver_name)
    for nfe in target_nfes:
        steps = max(1, nfe // cost)
        actual_nfe = steps * cost
        x_gen = solver_fn(params_tr, x0, steps=steps)
        w2 = empirical_w2_squared_hungarian(x_gen, x1)
        rows.append({'solver': solver_name, 'nfe': actual_nfe, 'steps': steps, 'endpoint_w2_sq': w2})

df = pd.DataFrame(rows)
df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for s in df['solver'].unique():
    d = df[df['solver'] == s].sort_values('nfe')
    ax.plot(d['nfe'], d['endpoint_w2_sq'], marker='o', label=s)
ax.set_xlabel('NFE (function evaluations)')
ax.set_ylabel(r'endpoint $W_2^2$')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.grid(alpha=0.2)
ax.legend()
plt.show()
